# Reproducing Vu et al. (2024) from dandiset 001084

**Paper.** Vu MT, Brown EH, Wen MJ, Noggle CA, Zhang Z, Monk KJ, Bouabid S, Mroz L, Graham BM, Zhuo Y,
Li Y, Otchy TM, Tian L, Davison IG, Boas DA, Howe MW (2024). Targeted micro-fiber arrays for measuring
and manipulating localized multi-scale neural dynamics over large, deep brain volumes during behavior.
*Neuron* 112, 909-923. [doi:10.1016/j.neuron.2023.12.011](https://doi.org/10.1016/j.neuron.2023.12.011).
This attempt used the full text of the NIH author manuscript in PubMed Central (PMC10957316). The
supplementary PDF could not be retrieved, so the supplementary figure legends were not read.

**Dandiset.** [DANDI:001084](https://dandiarchive.org/dandiset/001084/0.241023.2011), version
0.241023.2011. The paper cites no DANDI version (its data statement points to Zenodo). This is the only
published version, so it is pinned; the draft has the same 180 assets and size.

**Code read.** The paper's analysis code on Zenodo
([doi:10.5281/zenodo.10272874](https://doi.org/10.5281/zenodo.10272874)), which settles several choices
the Methods leave open (the 11-frame tone delay, the spline basis, the latency rule, how stimulation
times are found), and the conversion code
[catalystneuro/howe-lab-to-nwb](https://github.com/catalystneuro/howe-lab-to-nwb) at commit 3c53055.

**Outside inputs.** One claim (`fig4-link-published-results`) reads the paper's per-fiber results
(`glm_spline_results.mat`, Zenodo Fig04.zip) to check that the dandiset holds the data behind them.
Every other claim uses the dandiset alone.

Random numbers: each section that draws them has its own generator with a fixed seed.

| Claim | Data | Status |
|---|---|---|
| fig1c-isosbestic-control | dandiset | tested |
| figs2d-correlation-distance | dandiset | tested |
| figs2ef-no-bundle-crosstalk | dandiset | tested |
| fig1d-chronic-reward-stability | dandiset | tested |
| fig4-link-published-results | with inputs | tested |
| fig4e-light-increases-widespread | dandiset | tested |
| fig4e-light-anterior-posterior-gradient | dandiset | tested |
| fig4f-tone-tail-increase-anterior-decrease | dandiset | tested |
| fig4f-decreases-lateral-ventral | dandiset | tested |
| figs4ac-patterns-consistent-across-mice | dandiset | tested |
| figs4eh-patterns-not-movement | dandiset | tested |
| figs5-isosbestic-no-pattern | dandiset | tested |
| fig5ab-response-latencies | dandiset | tested |
| fig5g-light-latency-gradient | dandiset | tested |
| fig5h-tone-latency-gradient | dandiset | tested |
| figs6cd-gradients-individual-mice | dandiset | tested |
| fig6-inclusion-counts | dandiset | tested |
| fig6gh-release-decays-with-distance | dandiset | tested |
| fig6g-dorsal-spread | dandiset | tested |
| fig6-tdtomato-control-no-response | dandiset | tested |
| figs7a-photoswitch-confined | dandiset | tested |
| figs1-no-tissue-damage | Zenodo | blocked: not attempted (data only on Zenodo) |
| figs3-targeting-errors | Zenodo | blocked: not attempted (data only on Zenodo) |
| fig3-freely-moving-signals | Zenodo | blocked: not attempted (data only on Zenodo) |
| fig7d-ventral-not-dorsal-jaw | Zenodo | blocked: not attempted (data only on Zenodo) |
| fig7ef-single-ventral-fiber | Zenodo | blocked: not attempted (data only on Zenodo) |

In [ ]:
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")  # many small least-squares fits

import io
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.io as sio
import statsmodels.api as sm
from scipy import stats

import vu2024 as V

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
print("dandiset", V.DANDISET_ID, "version", V.DANDISET_VERSION)

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
r = requests.get(f"https://api.dandiarchive.org/api/dandisets/{V.DANDISET_ID}/versions/{V.DANDISET_VERSION}/")
r.raise_for_status()
meta = r.json()
summary = meta["assetsSummary"]
print("Archive summary: subjects", summary["numberOfSubjects"], "; files", summary["numberOfFiles"],
      "; species", [s["name"] for s in summary["species"]])
print("approach:", [a["name"] for a in summary["approach"]])
print("measurementTechnique:", [a["name"] for a in summary["measurementTechnique"]])
print("variableMeasured:", summary["variableMeasured"])
paths = V.list_nwb_paths()
print("\nNWB files:", len(paths))


def survey(path):
    f = V.open_nwb(path)
    sub = f["general/subject"]
    row = {"path": path, "subject": V._text(sub["subject_id"][()]), "genotype": V._text(sub["genotype"][()]),
           "sex": V._text(sub["sex"][()]), "session": V._text(f["general/session_id"][()]),
           "start": V._text(f["session_start_time"][()])[:19]}
    series = [k for k in f["acquisition"] if k.startswith("FiberPhotometryResponseSeries")]
    row["channels"] = ",".join(s.replace("FiberPhotometryResponseSeries", "") for s in series)
    g = f["acquisition"][series[0]]
    row["frames"] = g["data"].shape[0]
    row["rate_hz"] = round(float(g["starting_time"].attrs["rate"]), 2)
    row["fibers"] = g["data"].shape[1]
    tab = V.fiber_table(f, series[0])
    row["included"] = int(tab.included.sum())
    ip = f["general/optophysiology"]
    row["indicator"] = ",".join(sorted({V._text(ip[k]["indicator"][()]) for k in ip}))
    ev = V.read_events(f)
    for kind in ("Light", "Tone", "Reward", "Lick"):
        row[kind] = int((ev.event_type == kind).sum())
    row["videos"] = sum("Video" in k for k in f["acquisition"])
    return row


with ThreadPoolExecutor(16) as ex:
    files = pd.DataFrame(list(ex.map(survey, paths)))
files = files.sort_values(["subject", "session"]).reset_index(drop=True)
print("Distinct subjects in the files:", files.subject.nunique())
per_subject = files.groupby("subject").agg(
    sessions=("session", "size"), genotype=("genotype", "first"), sex=("sex", "first"),
    indicator=("indicator", "first"), channels=("channels", "first"), rate_hz=("rate_hz", "median"),
    fibers=("fibers", "first"), included=("included", "first"), light=("Light", "sum"), tone=("Tone", "sum"),
    reward=("Reward", "sum"), first=("start", "min"), last=("start", "max"))
print(per_subject.to_string())

The 16 subjects in the files match the archive's count. The archive lists the approach as "cell
population imaging"; the files hold fiber photometry (one trace per optical fiber) and the movies of
the fiber bundle it was extracted from. The subjects map onto the paper's experiments as follows, by genotype, indicator,
date and the paper's code:

- DL18, DL20, DL21, DL23, DL27, DL29, DL30: C57BL/6 with dLight1.3b, one light and tone session each
  (Figures 4, 5, S4, S6).
- DL15: dLight1.3b, 46 reward sessions over 8 weeks (Figure 1D).
- 842: D1-cre with GCaMP7f, 470 and 405 nm interleaved at 18 Hz (Figure 1C, S2D-F).
- DL31, DL32: dLight1.3b, 470 and 415 nm interleaved at 18 Hz, light and tone (Figure S5E-H).
- Grid-9: DAT-cre with Ach3.0 and jRGECO1a at 11 Hz (Figure S2C); no claim uses it.
- Opto2: DAT-cre with rDA3m (rDAm3.0), single-fiber stimulation sessions (Figure 6).
- Opto4, Opto9: DAT-cre with tdTomato, 13 stimulation sessions each, with face videos: the tdTomato
  controls of Figure 6 ("26 single-site stimulations across 2 mice").
- UG21: wild type, 3 stimulation sessions, indicator given as tdTomato. Its 3 sessions and 22
  included fibers match the paper's rDAm3.0 mouse without opsin ("0/22 non-stimulated sites for 3
  single-site stimulations"), and the paper's code names these sessions as the photoswitching
  controls of Figure S7A. The indicator in the files is consistent with being a mislabel; the files
  cannot show which indicator was expressed.

The D1-cre mice of Figure 7 (OptoD1_6, 7, 8, 10 in the paper's code), the freely moving mice of Figure
3, the histology of Figure S1, the CT scans of Figure 2, and the 470/405 nm sessions of Figure S5A-D
are not in the dandiset.

In [ ]:
def load_salient(mouse):
    f = V.open_nwb(V.SALIENT_SESSIONS[mouse])
    dff, times, tab = V.read_dff(f)
    beh = V.read_behavior(f)
    ev = V.read_events(f)
    fs = 1 / np.median(np.diff(beh["t"]))
    ev["on"] = V.nearest_index(beh["t"], ev.start_time.values)
    ev["off"] = V.nearest_index(beh["t"], ev.stop_time.values)
    return {"mouse": mouse, "sex": V._text(f["general/subject/sex"][()]), "dff_all": dff, "times": times,
            "table": tab, "beh": beh, "events": ev, "fs": fs}


with ThreadPoolExecutor(7) as ex:
    S = {d["mouse"]: d for d in ex.map(load_salient, V.SALIENT_SESSIONS)}

rows = []
for m, d in S.items():
    ev, tab, beh = d["events"], d["table"], d["beh"]
    light, tone = ev[ev.event_type == "Light"], ev[ev.event_type == "Tone"]
    d["dff"] = d["dff_all"][:, tab.included.values]
    d["fib"] = tab[tab.included].reset_index(drop=True)
    rows.append({
        "mouse": m, "sex": d["sex"], "frames": len(d["times"]), "behaviour samples": len(beh["t"]),
        "fs_hz": round(d["fs"], 3), "max |fiber t - behaviour t| ms": round(1000 * np.abs(d["times"] - beh["t"]).max(), 1),
        "fibers": len(tab), "included": int(tab.included.sum()), "light": len(light), "tone": len(tone),
        "light s": f"{(light.stop_time - light.start_time).min():.2f}-{(light.stop_time - light.start_time).max():.2f}",
        "tone s": f"{(tone.stop_time - tone.start_time).min():.2f}-{(tone.stop_time - tone.start_time).max():.2f}",
        "ITI s": f"{np.diff(np.sort(ev[ev.event_type.isin(['Light', 'Tone'])].start_time)).min():.1f}-"
                 f"{np.diff(np.sort(ev[ev.event_type.isin(['Light', 'Tone'])].start_time)).max():.1f}",
        "dff min": round(float(d["dff"].min()), 3), "dff max": round(float(d["dff"].max()), 3),
        "roll m/s": f"{beh['roll'].min():.2f}..{beh['roll'].max():.2f}",
        "pitch m/s": f"{beh['pitch'].min():.2f}..{beh['pitch'].max():.2f}",
        "angular rad/s": f"{beh['angular'].min():.1f}..{beh['angular'].max():.1f}"})
sal = pd.DataFrame(rows)
print(sal.to_string(index=False))
print("\nTotal included fibers:", sal.included.sum(), "; sexes:", sal.sex.value_counts().to_dict())
print("Paper: 280 fibers from 7 mice (4F + 3M); 14 to 21 presentations of each modality; ITI 4 to 40 s.")
locs = pd.concat([d["fib"] for d in S.values()]).location.value_counts()
print("\nLocations of the included fibers:", locs.to_dict())
excl = pd.concat([d["table"][~d["table"].included] for d in S.values()]).location.value_counts()
print("Locations of the other fibers (top 8):", excl.head(8).to_dict())
allfib = pd.concat([d["fib"].assign(mouse=m) for m, d in S.items()])
print("\nIncluded fiber coordinates (mm from bregma): AP %.2f to %.2f, ML %.2f to %.2f, DV %.2f to %.2f"
      % (allfib.AP.min(), allfib.AP.max(), allfib.ML.min(), allfib.ML.max(), allfib.DV.min(), allfib.DV.max()))
print("Fibers in the tail (AP <= -1.0 mm):", int((allfib.AP <= -1.0).sum()),
      "; per mouse:", allfib[allfib.AP <= -1.0].groupby("mouse").size().to_dict())

The included fibers number 280, from 4 female and 3 male mice, as in the paper; they are the fibers
the paper analysed (its code keeps fibers "in_striatum", and the per-mouse counts equal those of the
paper's results file, see `fig4-link-published-results`). The column describes them as "successfully
implanted"; the paper's criterion is a tip within or on the border of the striatum, and the included
fibers are labelled Caudoputamen, Nucleus accumbens or Striatum, while 87 Caudoputamen fibers are not
included. Fiber traces and behaviour have the same number of samples; the fiber series is stored with a
regular rate and the behaviour with timestamps, which differ by at most 25 ms, and every event starts
on a behaviour timestamp, so events are placed on frames by their behaviour sample, as the code does.
Velocity is in m/s and angular velocity in rad/s, with ranges plausible for a mouse on a ball. Six
mice have 18 to 21 presentations of each stimulus; DL18 has 14 light and 44 tone presentations. The
paper's code uses only 12 kHz tones and its results file has 14 for DL18; the files do not say which
of the 44 tones are 12 kHz. Light presentations last 0.7 to 1.0 s and tones about 1.33 s on the
trigger line; the code places the tone 11 frames after its trigger (speaker delay).

In [ ]:
# Clock check: decode light against tone from the population response after each onset, with the
# true onsets and with all onsets shifted by +-5 s (labels shifted in time).
TONE_SHIFT = 11  # frames at 30 Hz (paper's code)


def decode_accuracy_fs(dff, onsets, labels, shift, w):
    on = onsets + shift
    ok = (on - w >= 0) & (on + w + 1 <= len(dff))
    on, lab = on[ok], labels[ok]
    feats = np.stack([dff[o:o + w].mean(0) - dff[o - w:o].mean(0) for o in on])
    correct = 0
    for i in range(len(on)):  # leave-one-out nearest centroid on z-scored features
        tr = np.ones(len(on), bool)
        tr[i] = False
        mu, sd = feats[tr].mean(0), feats[tr].std(0) + 1e-9
        z = (feats - mu) / sd
        c0, c1 = z[tr & (lab == 0)].mean(0), z[tr & (lab == 1)].mean(0)
        pred = int(np.sum((z[i] - c1) ** 2) < np.sum((z[i] - c0) ** 2))
        correct += pred == lab[i]
    return correct / len(on), len(on)


def decode_accuracy(dff, onsets, labels, shift):
    return decode_accuracy_fs(dff, onsets, labels, shift, 30)


rows = []
for m, d in S.items():
    ev = d["events"]
    st = ev[ev.event_type.isin(["Light", "Tone"])]
    lab = (st.event_type == "Tone").values.astype(int)
    on = st.on.values + TONE_SHIFT * lab
    row = {"mouse": m}
    for shift in (0, 150, -150):
        acc, n = decode_accuracy(d["dff"], on, lab, shift)
        row[f"shift {shift / 30:+.0f} s"] = round(acc, 2)
    row["presentations"] = n
    row["chance (majority)"] = round(max(lab.mean(), 1 - lab.mean()), 2)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

With the true onsets the stimulus type is decoded from the fibers' responses well above chance in
every mouse, and with onsets shifted by 5 s it falls to about chance (for DL18, whose tones outnumber
lights 44 to 14, chance is 0.76). The event table and the fiber traces share one clock, as far as this
contrast can show.

In [ ]:
def load_two_channel(path):
    f = V.open_nwb(path)
    out = {"path": path, "beh": V.read_behavior(f), "events": V.read_events(f)}
    for color in ("Green", "GreenIsosbestic"):
        dff, times, tab = V.read_dff(f, color)
        out[color] = {"dff": dff[:, tab.included.values], "times": times, "fib": tab[tab.included].reset_index(drop=True),
                      "rate": 1 / np.median(np.diff(times))}
    seg = f["processing/ophys/ImageSegmentation/PlaneSegmentation"]
    out["centroids"] = seg["ROICentroids"][:]
    out["included_mask"] = V.fiber_table(f, "FiberPhotometryResponseSeriesGreen").included.values
    return out


with ThreadPoolExecutor(4) as ex:
    TWO = dict(zip(["842-221123", "842-221118", "DL31", "DL32"], ex.map(load_two_channel, [
        "sub-842/sub-842_ses-221123_ophys.nwb", "sub-842/sub-842_ses-221118_ophys.nwb",
        V.ISOSBESTIC_SESSIONS["DL31"], V.ISOSBESTIC_SESSIONS["DL32"]])))
for k, d in TWO.items():
    g, i = d["Green"], d["GreenIsosbestic"]
    ev = d["events"]
    print(f"{k}: 470 nm {g['dff'].shape} at {g['rate']:.2f} Hz from {g['times'][0]:.2f} s; isosbestic {i['dff'].shape}; "
          f"behaviour samples {len(d['beh']['t'])} from {d['beh']['t'][0]:.2f} s; ROI centroids {d['centroids'].shape}; "
          f"included {int(d['included_mask'].sum())}; events {ev.event_type.value_counts().to_dict()}; "
          f"DF/F 470 {g['dff'].min():.3f}..{g['dff'].max():.3f}, isosbestic {i['dff'].min():.3f}..{i['dff'].max():.3f}")
for k in ("DL31", "DL32"):
    d = TWO[k]
    g, i = d["Green"]["times"], d["GreenIsosbestic"]["times"]
    print(f"{k}: isosbestic frames lead 470 nm frames by {1000 * np.median(g[:len(i)] - i[:len(g)]):.1f} ms;"
          f" behaviour timestamps differ from the isosbestic series' regular frame times by up to"
          f" {1000 * np.abs(d['beh']['t'] - i[:len(d['beh']['t'])]).max():.1f} ms, a drift over the session")
    ev = d["events"]
    st = ev[ev.event_type.isin(["Light", "Tone"])].sort_values("start_time")
    lab = (st.event_type == "Tone").values.astype(int)
    on = V.nearest_index(d["beh"]["t"], st.start_time.values) + 6 * lab
    for color in ("Green", "GreenIsosbestic"):
        x = d[color]["dff"]
        accs = [round(decode_accuracy_fs(x, on, lab, sh, 18)[0], 2) for sh in (0, 90, -90)]
        print(f"   {color}: light against tone decoded, true onsets / +5 s / -5 s: {accs} (chance 0.5, {len(on)} presentations)")

Mouse 842 has 82 fibers per channel, 61 included; the paper's Figure 1C describes a 93-fiber array, and
its code for the figure reads this mouse's session of 2022-11-23. Its traces start 180 s into the
behaviour clock; no claim aligns them to events. DL31 and DL32 are recorded at 18 Hz with the two
wavelengths interleaved (415 nm frames 28 ms before the 470 nm frames), as the Methods describe; the
behaviour has one sample per frame pair (8177 for DL31), but its timestamps drift from the regular
frame times stored for the series by up to half a second over the session, so the regular rate does
not describe the frame times exactly. Events are therefore placed by their behaviour sample index, the
frame index the paper's code uses, for both channels. Light and tone are decoded from the 470 nm
signals better with these onsets (0.90 and 0.80) than with onsets shifted by 5 s (0.40 to 0.65); with 20
presentations per mouse the contrast is weak, so the alignment of these two sessions is consistent with
the data but not shown. The bundle-image ROI centroids give one row per fiber of a channel.

In [ ]:
dl15 = files[files.subject == "DL15"].copy()
dl15["date"] = pd.to_datetime(dl15.start.str[:10])
print(dl15[["session", "start", "frames", "rate_hz", "fibers", "included", "Reward", "Lick"]].to_string(index=False))
def load_dl15(path):
    f = V.open_nwb(path)
    dff, times, tab = V.read_dff(f)
    beh = V.read_behavior(f)
    ev = V.read_events(f)
    n = len(beh["t"])
    lick = np.zeros(n, bool)
    for _, e in ev[ev.event_type == "Lick"].iterrows():
        a, b = V.nearest_index(beh["t"], np.array([e.start_time, e.stop_time]))
        lick[a:b] = True
    rew = np.sort(V.nearest_index(beh["t"], ev[ev.event_type == "Reward"].start_time.values))
    nxt = np.r_[rew[1:] - 1, n - 1]
    cons = []
    for a, b in zip(rew, nxt):  # first lick after each reward and before the next (getRewInfo.m)
        k = np.flatnonzero(lick[a + 1:b + 1])
        if len(k):
            cons.append(a + 1 + k[0])
    cons = np.array([c for c in cons if c + 121 <= len(dff)], int)
    x = dff[:, tab.included.values]
    act = np.stack([x[c:c + 121] for c in cons]) if len(cons) else np.zeros((0, 121, x.shape[1]))
    return {"path": path, "start": V._text(f["session_start_time"][()])[:10], "n_rew": len(rew), "act": act,
            "n_cons": len(cons), "head": np.round(dff[:200], 6)}


with ThreadPoolExecutor(10) as ex:
    D15 = list(ex.map(load_dl15, dl15.path))
print("\nSessions with 25 rewards or fewer:", int((dl15.Reward <= 25).sum()), "from", dl15[dl15.Reward <= 25].date.min().date(),
      "to", dl15[dl15.Reward <= 25].date.max().date(), "; with more:", int((dl15.Reward > 25).sum()),
      "from", dl15[dl15.Reward > 25].date.min().date(), "to", dl15[dl15.Reward > 25].date.max().date())
print("Rewards followed by a lick before the next reward (consumption), per session:", [s_["n_cons"] for s_ in D15])

DL15 has 44 included fibers, as the 44 fibers of Figure 1D. Sessions up to 2021-09-13 have 10 to 18
rewards and nothing else; from 2021-09-10 on most sessions have 44 to 70 rewards. The paper's code
treats sessions with more than 25 rewards as a task with predicted rewards and keeps only the rewards
of one task phase (`virmen_phase`), a variable the files do not hold (the paper's behaviour file of
2021-09-14 on Zenodo has the ViRMEn task variables, read while preparing, not here). The session of
2021-09-21 has no reward.

In [ ]:
opto_paths = [p for p in paths if any(s in p for s in ("Opto2", "Opto4", "Opto9", "UG21"))]


def load_opto(path):
    f = V.open_nwb(path)
    color = "Red"
    dff, times, tab = V.read_dff(f, color)
    fs = 1 / np.median(np.diff(times))
    stim, n_art = V.stimulated_fiber(dff, fs)
    on, off = V.laser_epochs(dff[:, stim], fs) if stim is not None else (np.array([], int), np.array([], int))
    return {"path": path, "subject": path.split("/")[0][4:], "session": path.split("_ses-")[1].split("_")[0],
            "dff": dff, "fs": fs, "table": tab, "stim": stim, "n_artefact": n_art, "on": on, "off": off}


with ThreadPoolExecutor(12) as ex:
    OPTO = list(ex.map(load_opto, opto_paths))
op = pd.DataFrame([{
    "subject": o["subject"], "session": o["session"], "frames": o["dff"].shape[0], "fs": round(o["fs"], 2),
    "fibers": o["dff"].shape[1], "included": int(o["table"].included.sum()),
    "stim fiber (1-based)": None if o["stim"] is None else o["stim"] + 1,
    "stim included": None if o["stim"] is None else bool(o["table"].included[o["stim"]]),
    "stim location": None if o["stim"] is None else o["table"].location[o["stim"]],
    "fibers with artefact": o["n_artefact"], "epochs": len(o["on"]),
    "epoch s": f"{(np.median(o['off'] - o['on']) / o['fs']):.2f}" if len(o["on"]) else "",
    "interval s": f"{np.diff(o['on']).min() / o['fs']:.0f}-{np.diff(o['on']).max() / o['fs']:.0f}" if len(o["on"]) > 1 else "",
    "max dff stim": round(float(o["dff"][:, o["stim"]].max()), 2) if o["stim"] is not None else None} for o in OPTO])
print(op.to_string(index=False))

In [ ]:
# Recordings deposited twice: compare the first 200 DF/F frames of every session loaded here
# (all but Grid-9, the only session of its mouse).
heads = [(f"{m} salient", d["dff_all"][:200]) for m, d in S.items()]
heads += [(k, d["Green"]["dff"][:200]) for k, d in TWO.items()]
heads += [(f"DL15 {s_['start']}", s_["head"]) for s_ in D15]
heads += [(f"{o['subject']} {o['session']}", o["dff"][:200]) for o in OPTO]
same = [(a, b) for i, (a, x) in enumerate(heads) for b, y in heads[i + 1:] if x.shape == y.shape and np.allclose(x, y, equal_nan=True)]
print("Sessions compared:", len(heads), "; pairs with identical first 200 frames:", same if same else "none")

The files hold no stimulation times and no record of which fiber was stimulated. As in the paper's
code (`getStimTimes.m`), the laser epochs are found where the stimulated fiber's DF/F exceeds 0.4
(laser light leaking into the imaged fiber), here without the code's cross-check against the trigger
line, which the files do not hold. In Opto2 and UG21 exactly one fiber carries the artefact in each
session, except Opto2 2023-03-20f, where 15 fibers do, among them the midbrain fibers 1 to 5; it is
left out as a multi-site or midbrain stimulation. UG21's stimulated fibers are 12, 18 and 24, the
numbers of the paper's code folders for this mouse (`null_12`, `null_18`, `null_24`), and the first
three Opto2 sessions give fibers 13, 17 and 38, matching the code's `stri_13`, `stri_17` and `stri_38`
recorded on 2023-03-14. There are 7 to 11 epochs of about 1 s, 30 to 60 s apart, as the Methods
state. In the tdTomato mice (Opto4, Opto9) the artefact appears on many fibers at once; the fiber with
the largest one is taken as the stimulated fiber, which is not always included. Opto9 has 12
sessions, so the files hold 25 tdTomato stimulations where the paper reports 26.

## Salient-stimulus GLM (shared by Figures 4, 5 and S4 to S6)

For each of the 7 mice: overall velocity and acceleration from the treadmill; for each fiber the lags
of the most positive and most negative significant ITI cross-correlation with each (within 500 ms);
the GLM of the Methods on the -1 to 2 s windows; and the null of 10000 refits with random onsets and
shuffled labels. The Methods' cross-correlation is used; the code's version, in which negative lags
are not shifted, is shown as a variant of the point estimates.

In [ ]:
rng_glm = np.random.default_rng(10840)
N_NULL = 10000
t0 = time.time()
G = {}
for m, d in S.items():
    ev = d["events"]
    st = ev[ev.event_type.isin(["Light", "Tone"])].sort_values("start_time")
    lab = (st.event_type == "Tone").values.astype(int)
    on = st.on.values + TONE_SHIFT * lab
    fs = d["fs"]
    vel, acc = V.overall_velocity(d["beh"], fs)
    iti = V.iti_mask(len(vel), st.on.values, st.off.values, int(round(fs)))
    lags = V.loco_lags(d["dff"], vel, acc, iti, 15)
    glm = V.CueGLM(d["dff"], vel, acc, lags, fs)
    mu, regr = glm.fit(on, lab, keep_trials=True)
    null = V.cue_glm_null(glm, lab, N_NULL, rng_glm)
    lo, hi = np.percentile(null, [0.5, 99.5], axis=0)
    lags_code = V.loco_lags(d["dff"], vel, acc, iti, 15, code_indexing=True)
    mu_code, _ = V.CueGLM(d["dff"], vel, acc, lags_code, fs).fit(on, lab)
    G[m] = {"on": on, "lab": lab, "vel": vel, "acc": acc, "lags": lags, "mu": mu, "regr": regr, "lo": lo, "hi": hi,
            "mu_code": mu_code, "glm": glm}
    print(f"{m}: {len(on)} presentations ({(lab == 0).sum()} light, {(lab == 1).sum()} tone), {d['dff'].shape[1]} fibers; "
          f"locomotion regressors per fiber {np.bincount(glm.n_loco, minlength=5).tolist()} (0..4); "
          f"windows with NaN rows {int((~np.isfinite(glm._design(on, lab)[1]).all(1)).sum())}; {time.time() - t0:.0f} s")

rows = []
for m, g in G.items():
    fib = S[m]["fib"]
    for c, name in ((0, "light"), (1, "tone")):
        up = V.runs_of(g["mu"][c] > g["hi"][c])
        down = V.runs_of(g["mu"][c] < g["lo"][c])
        val, sig = V.signed_peak(g["mu"][c], up, down)
        valc, _ = V.signed_peak(g["mu_code"][c], up, down)
        for r in range(len(fib)):
            rows.append({"mouse": m, "fiber": r, "AP": fib.AP[r], "ML": fib.ML[r], "DV": fib.DV[r], "stim": name,
                         "value": val[r], "sig": bool(sig[r]), "up": bool(up[r]), "down": bool(down[r]),
                         "value_code_lags": valc[r]})
R = pd.DataFrame(rows)
W = R.pivot_table(index=["mouse", "fiber", "AP", "ML", "DV"], columns="stim",
                  values=["value", "sig", "up", "down", "value_code_lags"], aggfunc="first")
W.columns = [f"{a}_{b}" if b else a for a, b in W.columns]
W = W.reset_index()
for c in ("sig_light", "up_light", "down_light", "sig_tone", "up_tone", "down_tone"):
    W[c] = W[c].astype(bool)
TAIL = -1.0
W["is_tail"] = W.AP <= TAIL
print("\nfibers", len(W), "; tail", int(W.is_tail.sum()), f"; GLM and null took {time.time() - t0:.0f} s")
for name in ("light", "tone"):
    print(f"{name}: value range {W['value_' + name].min():.4f} to {W['value_' + name].max():.4f} DF/F; "
          f"correlation with the code-lag variant r = {np.corrcoef(W['value_' + name], W['value_code_lags_' + name])[0, 1]:.4f}")

## fig4-link-published-results

> The dandiset's DF/F traces, fibers and stimulus times are those behind the paper's salient-stimulus
> analysis, so that its per-fiber responses can be linked to the paper's.

Outside input: the paper's `glm_spline_results.mat` (Zenodo Fig04.zip), read by range requests.

In [ ]:
pub = sio.loadmat(io.BytesIO(V.zenodo_member("Fig04.zip", "Fig04/general_functions/glm_spline_results.mat")),
                  squeeze_me=True, struct_as_record=False)
rows = []
for m, g in G.items():
    p = pub[m]
    dff = S[m]["dff"]
    row = {"mouse": m, "fibers dandiset": dff.shape[1], "fibers paper": p.cue1_roi_mu.shape[1]}
    for c, name in ((0, "light"), (1, "tone")):
        ons = g["on"][g["lab"] == c]
        eta = np.nanmean(V.windows(dff, ons, -30, 60), 0)
        pm = getattr(p, f"cue{c + 1}_roi_mu")
        row[f"{name} trials"] = f"{len(ons)}/{getattr(p, f'cue{c + 1}_loco_regr_act').shape[2]}"
        row[f"{name} raw max|diff|"] = float(np.abs(eta - pm).max()) if eta.shape == pm.shape else np.nan
        row[f"{name} spline r"] = round(np.corrcoef(g["mu"][c].ravel(), getattr(p, f"cue{c + 1}_spline_mu").ravel())[0, 1], 4)
        psig = getattr(p, f"cue{c + 1}_spline_sig")
        up = V.runs_of(g["mu"][c] > g["hi"][c])
        down = V.runs_of(g["mu"][c] < g["lo"][c])
        row[f"{name} sig agree"] = round(float(np.mean((psig[:, 0] == up) & (psig[:, 1] == down))), 3)
    rows.append(row)
link = pd.DataFrame(rows)
print(link.to_string(index=False))
agree_all = sum((link[f"{n} sig agree"] * link["fibers dandiset"]).sum() for n in ("light", "tone")) / (2 * link["fibers dandiset"].sum())
print("\nSignificance agreement over all fiber and stimulus pairs:", round(float(agree_all), 3))
print("Spline estimates pooled over mice: light r = %.4f, tone r = %.4f" % (
    np.corrcoef(np.concatenate([G[m]["mu"][0].ravel() for m in G]), np.concatenate([pub[m].cue1_spline_mu.ravel() for m in G]))[0, 1],
    np.corrcoef(np.concatenate([G[m]["mu"][1].ravel() for m in G]), np.concatenate([pub[m].cue2_spline_mu.ravel() for m in G]))[0, 1]))

# DL18: identify the paper's 14 tones (12 kHz) among the 44 by matching its per-trial activity.
p = pub["DL18"]
dff = S["DL18"]["dff"]
tones = G["DL18"]["on"][G["DL18"]["lab"] == 1]
win = V.windows(dff, tones, -30, 60)  # tones x frames x fibers
pt = p.cue2_loco_regr_act  # frames x fibers x trials
match, best_r, second_r = [], [], []
for k in range(pt.shape[2]):
    r = np.array([np.ma.corrcoef(np.ma.masked_invalid(pt[:, :, k].ravel()), np.ma.masked_invalid(w.ravel()))[0, 1] for w in win])
    o = np.argsort(r)[::-1]
    match.append(o[0]); best_r.append(r[o[0]]); second_r.append(r[o[1]])
match = np.array(match)
print("\nDL18 tones matched to the paper's 14 trials:", (match + 1).tolist(), "; distinct:", len(set(match)))
print("match r: min %.3f; best competitor r: max %.3f" % (min(best_r), max(second_r)))
eta14 = np.nanmean(win[match], 0)
print("DL18 raw tone average over the 14 matched tones against the paper's: max |diff| = %.2e" % np.abs(eta14 - p.cue2_roi_mu).max())
DL18_12K = tones[np.sort(match)]

**Result: reproduced.** For every mouse the raw triggered averages computed from the dandiset equal
the paper's published ones (largest difference 0.0) for light, and for tone in the six mice other than
DL18, whose file holds 44 tones where the paper's results use 14. The GLM spline estimates correlate with
the published ones at r = 0.9957 for light and 0.9922 for tone over all fibers, and the significance of
increases and decreases agrees for 93.6% of fiber and stimulus pairs (lowest: DL18 tone 0.684, with 44
tones in place of 14; DL29 light 0.822). DL18's 14 tones are identified one to one (match r at least
0.952, best competing tone at most 0.693), and their raw average equals the published one. The dandiset's
traces, fiber order and stimulus times are those of the paper's analysis; only the tone frequency of DL18
is missing.

## fig4e-light-increases-widespread

> The light evokes transient increases in dopamine release across many striatal locations.

In [ ]:
n = len(W)
up, down = int(W.up_light.sum()), int(W.down_light.sum())
val_up = int((W.sig_light & (W.value_light > 0)).sum())
val_down = int((W.sig_light & (W.value_light < 0)).sum())
print(f"Light: fibers with a significant increase {up}/{n} ({up / n:.0%}), with a significant decrease {down}/{n};"
      f" by the signed largest value: {val_up} increases, {val_down} decreases, {n - val_up - val_down} not significant")
print(W.groupby("mouse").agg(fibers=("fiber", "size"), increase=("up_light", "sum"), decrease=("down_light", "sum")).to_string())
print("Null: each fiber is tested on 31 frames against a 0.5 to 99.5 percentile band; the fraction of fibers that"
      " reach 3 consecutive frames by chance is estimated below from 200 of the null refits per mouse.")
fp = []
for m, g in G.items():
    sub = V.cue_glm_null(g["glm"], g["lab"], 200, np.random.default_rng(1))
    fp.append(np.mean([V.runs_of(sub[i, 0] > g["hi"][0]).mean() for i in range(200)]))
print("chance rate of a significant light increase per fiber (mean over mice): %.4f" % np.mean(fp))

**Result: reproduced.** 240 of 280 fibers (86%) have a significant increase to light and 38 a
significant decrease; by the signed largest value, 237 are increases and 11 decreases. Every mouse has
increases in most fibers (from 34 of 51 in DL21 to 50 of 53 in DL20), and decreases are concentrated in DL21 (22 of 51) and
DL29 (12 of 45). The test's chance rate of a significant increase per fiber, from null refits, is 0.024.

## fig4e-light-anterior-posterior-gradient

> Light-evoked dopamine increases follow an amplitude gradient from the anterior striatum into the
> posterior tail, where increases are less prevalent and smaller.

In [ ]:
def light_gradient(w, tail):
    rho, p = stats.spearmanr(w.AP, w.value_light)
    t = w.AP <= tail
    table = [[int((w.up_light & t).sum()), int((~w.up_light & t).sum())],
             [int((w.up_light & ~t).sum()), int((~w.up_light & ~t).sum())]]
    _, p_frac = stats.fisher_exact(table, alternative="less")
    _, p_amp = stats.mannwhitneyu(w.value_light[t], w.value_light[~t], alternative="less")
    return {"tail AP <=": tail, "n tail": int(t.sum()), "rho AP-light": round(rho, 3), "p rho": p,
            "frac up tail": round(table[0][0] / max(t.sum(), 1), 2), "frac up other": round(table[1][0] / (~t).sum(), 2),
            "p frac": p_frac, "median tail": round(w.value_light[t].median(), 4),
            "median other": round(w.value_light[~t].median(), 4), "p amp": p_amp}


rows = [light_gradient(W, t) for t in (TAIL, -0.8, -1.3)]
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer mouse (tail AP <= -1.0):")
print(pd.DataFrame([{"mouse": m, **light_gradient(W[W.mouse == m], TAIL)} for m in G]).to_string(index=False))
rho_c, p_c = stats.spearmanr(W.AP, W.value_code_lags_light)
print(f"\nVariant, code's lags: rho(AP, light) = {rho_c:.3f}, p = {p_c:.2g}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, s in zip(ax, ("light", "tone")):
    a.scatter(W.AP, W["value_" + s], c=np.where(W["sig_" + s], "k", "0.7"), s=10)
    a.axvline(TAIL, ls="--", c="r")
    a.axhline(0, c="0.5", lw=0.5)
    a.set_xlabel("AP (mm from bregma)")
    a.set_ylabel(f"{s}: signed largest DF/F")
plt.tight_layout()
plt.show()
print("Plotted ranges: AP %.2f to %.2f; light %.4f to %.4f; tone %.4f to %.4f (all points shown, black = significant)"
      % (W.AP.min(), W.AP.max(), W.value_light.min(), W.value_light.max(), W.value_tone.min(), W.value_tone.max()))

**Result: partially reproduced.** The light value rises from posterior to anterior (Spearman rho =
0.168, p = 0.0047) and is smaller in the tail (median 0.0055 against 0.0117 DF/F, p = 0.0013), but
increases are not less prevalent there: 83% of tail fibers against 86% elsewhere have a significant
increase (p = 0.38). The same holds with the tail at -0.8 or -1.3 mm and with the code's lags (rho =
0.162). The gradient is carried by DL18, DL20 and DL27 (rho 0.45 to 0.69); three mice have a negative
point estimate (DL23 -0.43, DL29 -0.25, DL30 -0.12).

## fig4f-tone-tail-increase-anterior-decrease

> The tone evokes the strongest dopamine increases at sites in the posterior tail, while anterior to the
> tail it evokes primarily decreases.

In [ ]:
def tone_pattern(w, tail, col="value_tone"):
    t = w.AP <= tail
    _, p_amp = stats.mannwhitneyu(w[col][t], w[col][~t], alternative="greater")
    top = w.sort_values(col, ascending=False).head(10)
    ant = w[~t & w.sig_tone]
    n_dec, n_inc = int((ant[col] < 0).sum()), int((ant[col] > 0).sum())
    p_bin = stats.binomtest(n_dec, n_dec + n_inc, 0.5, alternative="greater").pvalue if n_dec + n_inc else np.nan
    return {"tail AP <=": tail, "n tail": int(t.sum()), "median tail": round(w[col][t].median(), 4),
            "median anterior": round(w[col][~t].median(), 4), "p tail>anterior": p_amp,
            "top10 in tail": int((top.AP <= tail).sum()), "anterior sig dec": n_dec, "anterior sig inc": n_inc,
            "p binomial": p_bin}


print(pd.DataFrame([tone_pattern(W, t) for t in (TAIL, -0.8, -1.3)]).to_string(index=False))
print("\nTail fibers with a significant tone increase / decrease:", int((W.is_tail & W.up_tone).sum()), "/", int((W.is_tail & W.down_tone).sum()),
      "of", int(W.is_tail.sum()), "; anterior:", int((~W.is_tail & W.up_tone).sum()), "/", int((~W.is_tail & W.down_tone).sum()), "of", int((~W.is_tail).sum()))
print("\nPer mouse:")
print(pd.DataFrame([{"mouse": m, **tone_pattern(W[W.mouse == m], TAIL)} for m in G]).to_string(index=False))
print("\nVariant, code's lags:", tone_pattern(W.assign(value_tone=W.value_code_lags_tone), TAIL))

# Variant: DL18 with only the 12 kHz tones identified from the paper's results (fig4-link-published-results)
g = G["DL18"]
keep = (g["lab"] == 0) | np.isin(g["on"], DL18_12K)
mu18, _ = g["glm"].fit(g["on"][keep], g["lab"][keep])
null18 = V.cue_glm_null(g["glm"], g["lab"][keep], N_NULL, np.random.default_rng(18))
lo18, hi18 = np.percentile(null18, [0.5, 99.5], axis=0)
v18, s18 = V.signed_peak(mu18[1], V.runs_of(mu18[1] > hi18[1]), V.runs_of(mu18[1] < lo18[1]))
W12 = W.copy()
W12.loc[W12.mouse == "DL18", "value_tone"] = v18
W12.loc[W12.mouse == "DL18", "sig_tone"] = s18
print("Variant, DL18 with its 14 12-kHz tones (with inputs):", tone_pattern(W12, TAIL))

**Result: reproduced.** Tail fibers have larger tone values than anterior fibers (median 0.0092
against -0.0026 DF/F, p = 5.5e-13), all 10 of the largest tone values are in the tail, and 32 of 35 tail
fibers have a significant increase with none decreasing. Anterior to the tail, 103 fibers with a
significant tone response decrease and 60 increase (binomial p = 0.00047). The tail comparison holds at
every tail boundary and with the code's lags, and the result is the same with DL18 restricted to its 14
12-kHz tones (108 decreases against 60). The anterior decrease is not uniform across mice: it is
significant in DL21 (36 against 3) and DL23 (17 against 6), while DL18, DL20, DL27 and DL30 have as many
or more anterior increases.

## fig4f-decreases-lateral-ventral

> Tone-evoked decreases are slightly more prevalent in the anterior lateral and ventral striatum, with
> many sites showing both increases to the light and decreases to the tone.

In [ ]:
A = W[~W.is_tail].copy()
A["tone_dec"] = A.sig_tone & (A.value_tone < 0)
rows = []
for coord in ("ML", "DV", "AP"):
    _, p = stats.mannwhitneyu(A[coord][A.tone_dec], A[coord][~A.tone_dec], alternative="greater")
    rows.append({"coordinate": coord, "median, tone decrease": A[coord][A.tone_dec].median(),
                 "median, other": A[coord][~A.tone_dec].median(), "p (decrease > other)": p})
print("Anterior fibers:", len(A), "; with a significant tone decrease:", int(A.tone_dec.sum()))
print(pd.DataFrame(rows).to_string(index=False))
both = int((A.sig_light & (A.value_light > 0) & A.tone_dec).sum())
print(f"\nAnterior fibers with a significant light increase and a significant tone decrease: {both}/{len(A)} ({both / len(A):.0%})")
for tail in (-0.8, -1.3):
    B = W[W.AP > tail].copy()
    B["tone_dec"] = B.sig_tone & (B.value_tone < 0)
    ps = [stats.mannwhitneyu(B[c][B.tone_dec], B[c][~B.tone_dec], alternative="greater")[1] for c in ("ML", "DV")]
    print(f"Variant tail AP <= {tail}: p ML = {ps[0]:.3g}, p DV = {ps[1]:.3g}")

**Result: partially reproduced.** Among 245 anterior fibers, the 103 with a significant tone decrease
lie more ventrally than the others (median DV 3.66 against 3.30 mm, p = 0.0031) but not more laterally
(median ML 2.07 against 1.97 mm, p = 0.43); the ventral effect holds at both other tail boundaries and the
lateral one at neither. They also lie more anteriorly (p = 1e-06). 79 anterior fibers (32%) have both a
significant light increase and a significant tone decrease, so the second part of the statement holds by
the 20% reading.

## figs4ac-patterns-consistent-across-mice

> The modality-specific release patterns are consistent within and across mice.

In [ ]:
rows = []
for m in G:
    w = W[W.mouse == m]
    rl, pl = stats.spearmanr(w.AP, w.value_light)
    rt, pt_ = stats.spearmanr(w.AP, w.value_tone)
    rows.append({"mouse": m, "fibers": len(w), "AP range": f"{w.AP.min():.2f}..{w.AP.max():.2f}", "tail fibers": int(w.is_tail.sum()),
                 "rho AP-light": round(rl, 3), "p light": round(pl, 4), "rho AP-tone": round(rt, 3), "p tone": round(pt_, 4)})
pm = pd.DataFrame(rows)
print(pm.to_string(index=False))
print(f"\nMice with a positive AP-light correlation: {(pm['rho AP-light'] > 0).sum()}/7;"
      f" with a negative AP-tone correlation: {(pm['rho AP-tone'] < 0).sum()}/7")

**Result: partially reproduced.** The tone pattern is consistent across mice: AP and the tone value
are negatively correlated in all 7 mice (rho -0.44 to -0.70, each p < 0.03). The light gradient is not:
the correlation is positive in 4 of 7 mice (DL18, DL20, DL21, DL27; significant in three) and negative in
DL23 (significant), DL29 and DL30.

## figs4eh-patterns-not-movement

> Locomotor changes to the stimuli are variable and bi-directional, while the dopamine release patterns to
> light and tone are consistent, so the patterns are stimulus and not movement driven.

In [ ]:
rows, split = [], []
for m, g in G.items():
    dff = S[m]["dff"]
    for c, name in ((0, "light"), (1, "tone")):
        ons = g["on"][g["lab"] == c]
        acc = V.windows(g["acc"], ons, 0, 30)[:, :, 0]
        k = np.nanargmax(np.abs(acc), axis=1)
        sign = np.sign(acc[np.arange(len(ons)), k])
        dv = np.nanmean(V.windows(g["vel"], ons, 0, 30)[:, :, 0], 1) - np.nanmean(V.windows(g["vel"], ons, -30, -1)[:, :, 0], 1)
        rows.append({"mouse": m, "stim": name, "acc trials": int((sign > 0).sum()), "dec trials": int((sign < 0).sum()),
                     "mean velocity change m/s": round(dv.mean(), 4), "velocity up/down": f"{(dv > 0).sum()}/{(dv < 0).sum()}"})
        for part, mask in (("acc", sign > 0), ("dec", sign < 0)):
            if mask.sum() > 1:
                mu = np.nanmean(V.windows(dff, ons[mask], 0, 30), 0)
                v = np.where(np.abs(mu.min(0)) > np.abs(mu.max(0)), mu.min(0), mu.max(0))
            else:
                v = np.full(dff.shape[1], np.nan)
            for r in range(dff.shape[1]):
                split.append({"mouse": m, "fiber": r, "stim": name, "part": part, "value": v[r]})
print(pd.DataFrame(rows).to_string(index=False))
SP = pd.DataFrame(split)
SP["col"] = SP.stim + "_" + SP.part
SP = SP.pivot_table(index=["mouse", "fiber"], columns="col", values="value", dropna=False).reset_index()
SP.columns.name = None
SP = SP.merge(W[["mouse", "fiber", "AP", "is_tail"]], on=["mouse", "fiber"])
for name in ("light", "tone"):
    a, d = SP[f"{name}_acc"], SP[f"{name}_dec"]
    ok = a.notna() & d.notna()
    print(f"\n{name}: fibers with both splits {ok.sum()}; r(acc, dec) = {np.corrcoef(a[ok], d[ok])[0, 1]:.3f}")
    for part in ("acc", "dec"):
        v = SP[f"{name}_{part}"]
        print(f"  {part}: tail median {v[SP.is_tail].median():.4f}, anterior median {v[~SP.is_tail].median():.4f}")

**Result: partially reproduced.** Locomotion after the stimuli varies across mice and presentations:
the split into presentations followed by acceleration and by deceleration ranges from 0 and 21 (DL29
light) to 21 and 23 (DL18 tone), and the mean velocity change is positive in some mice and negative in
others for both stimuli. The fiber maps of the two splits correlate (light r = 0.873 over 235 fibers, tone
r = 0.784 over 229), and the light map has positive medians in both splits. The tone pattern holds after
acceleration (tail median 0.0073, anterior median -0.0031) but not after deceleration, where the anterior
median is positive (0.0042, tail 0.0098). This test uses raw DF/F, as the paper's code does, without the
locomotion regressors of the GLM.

## figs5-isosbestic-no-pattern

> With isosbestic (415 nm) excitation, large fluorescence changes and spatial patterns to light and tone
> are not observed, ruling out a hemodynamic origin of the release patterns.

In [ ]:
rng_s5 = np.random.default_rng(5415)
rows, S5 = [], []
for m in ("DL31", "DL32"):
    d = TWO[m]
    beh, ev = d["beh"], d["events"]
    st = ev[ev.event_type.isin(["Light", "Tone"])].sort_values("start_time")
    for color, label in (("Green", "470"), ("GreenIsosbestic", "415")):
        ch = d[color]
        fs = ch["rate"]
        n_fr = len(ch["dff"])
        b2 = {k: beh[k][:n_fr] for k in ("roll", "pitch", "angular")}  # behaviour sample k = frame pair k
        vel, acc = V.overall_velocity(b2, fs)
        lab = (st.event_type == "Tone").values.astype(int)
        on = V.nearest_index(beh["t"], st.start_time.values) + 6 * lab  # 6-frame tone delay at 18 Hz (code)
        off = V.nearest_index(beh["t"], st.stop_time.values)
        iti = V.iti_mask(len(vel), on - 6 * lab, off, int(round(fs)))
        lags = V.loco_lags(ch["dff"], vel, acc, iti, 9)
        glm = V.CueGLM(ch["dff"], vel, acc, lags, fs)
        mu, _ = glm.fit(on, lab)
        null = V.cue_glm_null(glm, lab, N_NULL, rng_s5)
        lo, hi = np.percentile(null, [0.5, 99.5], axis=0)
        for c, name in ((0, "light"), (1, "tone")):
            up, down = V.runs_of(mu[c] > hi[c]), V.runs_of(mu[c] < lo[c])
            val, sig = V.signed_peak(mu[c], up, down)
            fib = ch["fib"]
            rows.append({"mouse": m, "nm": label, "stim": name, "fibers": len(val), "trials": int((lab == c).sum()),
                         "median |value|": round(float(np.median(np.abs(val))), 4), "significant": int(sig.sum()),
                         "rho AP-value": round(stats.spearmanr(fib.AP, val)[0], 3)})
            S5.append(pd.DataFrame({"mouse": m, "nm": label, "stim": name, "fiber": np.arange(len(val)), "value": val}))
s5 = pd.DataFrame(rows)
print(s5.to_string(index=False))
allv = pd.concat(S5)
for name in ("light", "tone"):
    a = allv[(allv.stim == name) & (allv.nm == "470")].value.abs().median()
    b = allv[(allv.stim == name) & (allv.nm == "415")].value.abs().median()
    n470 = s5[(s5.stim == name) & (s5.nm == "470")].significant.sum()
    n415 = s5[(s5.stim == name) & (s5.nm == "415")].significant.sum()
    print(f"{name}: median |value| 470 nm {a:.4f}, 415 nm {b:.4f} (ratio {b / a:.2f}); significant fibers 470 {n470}, 415 {n415}")

**Result: partially reproduced.** With 415 nm excitation almost no fiber responds significantly (light
2 against 45 at 470 nm; tone 1 against 18 over 96 fibers of two mice), and the AP correlations seen at 470
nm for tone (-0.49 and -0.43) are not present at 415 nm (-0.19 and 0.16). The median absolute value at
415 nm is 0.49 of that at 470 nm for light, below the criterion's half, but 0.68 for tone, where most 470
nm fibers are not significant either and both medians are near the noise level. The 470 and 405 nm
sessions of Figure S5A-D are not in the dandiset.

## fig5ab-response-latencies

> Latencies to half-maximum are shorter for dopamine increases to light than to tone, and shorter for
> increases than decreases to tone (light increase 221.6 ms, SD 42.5; tone increase 305.4 ms, SD 100.0;
> tone decrease 428.3 ms, SD 197.1).

In [ ]:
FS = 30.0
lat = []
for m, g in G.items():
    for c, name in ((0, "light"), (1, "tone")):
        up = V.runs_of(g["mu"][c] > g["hi"][c])
        down = V.runs_of(g["mu"][c] < g["lo"][c])
        for r in range(g["mu"].shape[2]):
            if up[r]:
                lat.append({"mouse": m, "fiber": r, "stim": name, "kind": "increase", "frames": V.half_max_latency(g["mu"][c][:, r], 1)})
            if down[r]:
                lat.append({"mouse": m, "fiber": r, "stim": name, "kind": "decrease", "frames": V.half_max_latency(g["mu"][c][:, r], -1)})
L = pd.DataFrame(lat)
L["ms (Methods)"] = 1000 * L.frames / FS
L["ms (code, +1 frame)"] = 1000 * (L.frames + 1) / FS
summ = L.groupby(["stim", "kind"]).agg(n=("frames", "size"), no_crossing=("frames", lambda x: int(x.isna().sum())),
                                      mean_ms=("ms (Methods)", "mean"), sd_ms=("ms (Methods)", "std"),
                                      mean_code_ms=("ms (code, +1 frame)", "mean"), sd_code_ms=("ms (code, +1 frame)", "std"))
print(summ.round(1).to_string())
print("Paper: light increase 221.6 (SD 42.5); tone increase 305.4 (SD 100.0); tone decrease 428.3 (SD 197.1) ms")
print("\nPer mouse, mean ms (Methods):")
print(L.groupby(["mouse", "stim", "kind"])["ms (Methods)"].agg(["size", "mean"]).round(1).unstack(["stim", "kind"]).to_string())
fig, ax = plt.subplots(figsize=(6, 3))
for (s, k), col in ((("light", "increase"), "C1"), (("tone", "increase"), "C3"), (("tone", "decrease"), "C0")):
    ax.hist(L[(L.stim == s) & (L.kind == k)]["ms (Methods)"].dropna(), bins=np.arange(0, 1034, 1000 / 30), alpha=0.5, label=f"{s} {k}", color=col)
ax.set_xlabel("latency to half max (ms, from onset frame)")
ax.legend()
plt.show()
print("Plotted range %.0f to %.0f ms; all values within the axes" % (L["ms (Methods)"].min(), L["ms (Methods)"].max()))

**Result: partially reproduced.** The order of the latencies holds (light increase 187.5 ms < tone
increase 271.7 ms < tone decrease 393.2 ms, counted from the onset frame as the Methods define it), and
the tone decrease is within tolerance, but the light increase (expected 221.6 +- 22.2) and the tone
increase (305.4 +- 30.5) are below it. Counted as the paper's code counts, one frame later, the means are
220.8, 305.0 and 426.5 ms, within 2 ms of the paper's values, with SDs of 38.1, 98.7 and 196.5 ms against
42.5, 100.0 and 197.1. The difference from the paper is the code's one-frame offset. One tone increase has
no half-max crossing before its peak.

## fig5g-light-latency-gradient

> Dopamine release latencies to light follow an anterior-posterior gradient, with faster increases in the
> anterior striatum, consistent with wave-like dynamics.

In [ ]:
def trial_latencies(m, c, kind):
    """Per presentation and fiber, half-max latency (frames) of the trial activity with locomotion
    regressed out, where the fiber's mean response is significant and the trial exceeds the null band
    for 3 consecutive frames (rise_dip_time.m)."""
    g = G[m]
    tau = g["glm"].tau
    w = (tau >= 0) & (tau <= 30)
    act = g["regr"][g["lab"] == c][:, w, :]  # trials x 31 x fibers
    up = V.runs_of(g["mu"][c] > g["hi"][c])
    down = V.runs_of(g["mu"][c] < g["lo"][c])
    out = np.full((act.shape[0], act.shape[2]), np.nan)
    for t in range(act.shape[0]):
        for r in range(act.shape[2]):
            x = act[t, :, r]
            if kind == "increase" and up[r] and V.runs_of(x > g["hi"][c][:, r])[0]:
                out[t, r] = V.half_max_latency(x, 1)
            if kind == "decrease" and down[r] and V.runs_of(x < g["lo"][c][:, r])[0]:
                out[t, r] = V.half_max_latency(x, -1)
    return out


REL = []
for m in G:
    rise_l = trial_latencies(m, 0, "increase")
    rise_t = trial_latencies(m, 1, "increase")
    dip_t = trial_latencies(m, 1, "decrease")
    either_t = np.fmin(rise_t, dip_t)
    for name, mat in (("light rise", rise_l), ("tone either", either_t), ("tone rise", rise_t), ("tone dip", dip_t)):
        rel = mat - np.nanmin(mat, axis=1, keepdims=True)
        for r in range(mat.shape[1]):
            n = int(np.isfinite(rel[:, r]).sum())
            REL.append({"mouse": m, "fiber": r, "measure": name, "n_trials": n,
                        "rel_ms": 1000 * np.nanmean(rel[:, r]) / FS if n else np.nan,
                        "abs_ms": 1000 * np.nanmean(mat[:, r]) / FS if n else np.nan})
REL = pd.DataFrame(REL).merge(W[["mouse", "fiber", "AP", "ML", "DV"]], on=["mouse", "fiber"])


def gradient(measure, sign_word):
    d = REL[(REL.measure == measure) & REL.rel_ms.notna()]
    rho, p = stats.spearmanr(d.AP, d.rel_ms)
    print(f"{measure}: fibers with a latency {len(d)} (trials per fiber median {d.n_trials.median():.0f});"
          f" Spearman rho(AP, relative latency) = {rho:.3f}, p = {p:.2g} ({sign_word})")
    for c in ("ML", "DV"):
        r2, p2 = stats.spearmanr(d[c], d.rel_ms)
        print(f"   rho({c}, relative latency) = {r2:.3f}, p = {p2:.2g}")
    return d, rho, p


d_light, rho_l, p_l = gradient("light rise", "negative = anterior faster")
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(d_light.AP, d_light.rel_ms, s=10)
ax.set_xlabel("AP (mm)")
ax.set_ylabel("mean relative latency, light (ms)")
plt.show()
print("Plotted ranges: AP %.2f to %.2f; latency %.0f to %.0f ms" % (d_light.AP.min(), d_light.AP.max(), d_light.rel_ms.min(), d_light.rel_ms.max()))

**Result: reproduced.** Over the 240 fibers with a light latency, the mean latency relative to the
fastest fiber of each presentation is shorter anteriorly (Spearman rho(AP, latency) = -0.285, p = 7e-06).
Latency also increases laterally (rho(ML) = 0.263, p = 3.8e-05) and not with depth (p = 0.93).

## fig5h-tone-latency-gradient

> Dopamine release latencies to tone follow a different gradient, faster in the posterior striatum than
> in the anterior striatum.

In [ ]:
d_tone, rho_t, p_t = gradient("tone either", "positive = posterior faster")
for meas in ("tone rise", "tone dip"):
    gradient(meas, "component")
t = d_tone.AP <= TAIL
print(f"Tone relative latency: tail median {d_tone.rel_ms[t].median():.0f} ms (n {t.sum()}), anterior median {d_tone.rel_ms[~t].median():.0f} ms (n {(~t).sum()})")

**Result: reproduced.** Over the 196 fibers with a tone latency (the earlier of increase and
decrease), relative latency is shorter posteriorly (rho(AP, latency) = 0.254, p = 0.00033; tail median
308 ms against 368 ms anterior). Taken separately, neither increases (rho = -0.07, p = 0.5) nor decreases
(rho = -0.21, p = 0.028, the opposite sign) show this gradient, so it comes from the tail's fast increases
and the anterior striatum's slower decreases, as the paper states for Figure S6A-B.

## figs6cd-gradients-individual-mice

> The spatial latency gradients are also present in individual mice.

In [ ]:
rows = []
for m in G:
    row = {"mouse": m}
    for meas, key in (("light rise", "light"), ("tone either", "tone")):
        d = REL[(REL.mouse == m) & (REL.measure == meas) & REL.rel_ms.notna()]
        row[f"{key} fibers"] = len(d)
        if len(d) >= 5:
            rho, p = stats.spearmanr(d.AP, d.rel_ms)
            row[f"{key} rho"], row[f"{key} p"] = round(rho, 3), round(p, 4)
    rows.append(row)
pmice = pd.DataFrame(rows)
print(pmice.to_string(index=False))
print(f"\nLight: negative in {(pmice['light rho'] < 0).sum()} of {pmice['light rho'].notna().sum()} mice with >= 5 fibers;"
      f" tone: positive in {(pmice['tone rho'] > 0).sum()} of {pmice['tone rho'].notna().sum()}")

**Result: partially reproduced.** The tone gradient is positive in 6 of 7 mice (significant in DL20
and DL23); the light gradient is negative in 4 of 7 (significant in DL18 and DL27), with DL21 at 0 and
DL23 and DL29 positive.

## fig1c-isosbestic-control

> In a D1-cre mouse expressing GCaMP7f in D1 spiny projection neurons, fluorescence under 470 nm excitation
> shows Ca2+ transients across fibers that are absent under 405 nm (isosbestic) excitation.

In [ ]:
for key in ("842-221123", "842-221118"):
    d = TWO[key]
    g, i = d["Green"]["dff"], d["GreenIsosbestic"]["dff"]
    n = min(len(g), len(i))
    ratio = g[:n].std(0) / i[:n].std(0)
    print(f"{key}: fibers {len(ratio)}; SD ratio 470/405 median {np.median(ratio):.2f}, range {ratio.min():.2f} to {ratio.max():.2f};"
          f" ratio > 1 in {(ratio > 1).sum()}/{len(ratio)} ({(ratio > 1).mean():.0%}); 470 SD median {np.median(g.std(0)):.4f},"
          f" 405 SD median {np.median(i.std(0)):.4f}")
    big = (g[:n] > 5 * np.median(i[:n].std(0))).mean(0)
    print(f"   fraction of frames above 5 x the median 405 SD: 470 median {np.median(big):.4f};"
          f" 405 median {np.median((i[:n] > 5 * np.median(i[:n].std(0))).mean(0)):.4f}")

**Result: reproduced.** In all 61 included fibers of mouse 842 the DF/F under 470 nm varies more than
under 405 nm (SD ratio median 3.01, range 1.56 to 19.41). Frames above 5 times the median 405 nm SD make
up a median 6.7% of each 470 nm trace and none of the 405 nm traces. The other session gives the same
(median ratio 2.84, 61 of 61 above 1).

## figs2d-correlation-distance

> Pairwise correlations of D1 SPN Ca2+ signals are generally low but variable, and drop off significantly
> with the distance between fiber tips.

In [ ]:
d = TWO["842-221123"]
g = d["Green"]["dff"]
fib = d["Green"]["fib"]
cent = d["centroids"][d["included_mask"]]
C = np.corrcoef(g.T)
iu = np.triu_indices(len(fib), 1)
tip = fib[["ML", "AP", "DV"]].values
pairs = pd.DataFrame({"corr": C[iu],
                      "euc": np.linalg.norm(tip[:, None] - tip[None], axis=2)[iu],
                      "AP": np.abs(tip[:, None, 1] - tip[None, :, 1])[iu],
                      "ML": np.abs(tip[:, None, 0] - tip[None, :, 0])[iu],
                      "DV": np.abs(tip[:, None, 2] - tip[None, :, 2])[iu],
                      "surface": np.linalg.norm(cent[:, None] - cent[None], axis=2)[iu]})
r, p = stats.pearsonr(pairs["corr"], pairs.euc)
print(f"pairs {len(pairs)}; pairwise r median {pairs['corr'].median():.3f}, IQR {pairs['corr'].quantile(0.25):.3f} to"
      f" {pairs['corr'].quantile(0.75):.3f}, range {pairs['corr'].min():.3f} to {pairs['corr'].max():.3f}")
print(f"correlation of pairwise r with tip distance: r = {r:.3f}, p = {p:.2g}; Spearman {stats.spearmanr(pairs['corr'], pairs.euc)[0]:.3f}")
q = pairs.euc <= pairs.euc.quantile(0.25)
print(f"closest quarter of pairs (<= {pairs.euc.quantile(0.25):.2f} mm): mean r {pairs['corr'][q].mean():.3f}; farthest quarter mean r"
      f" {pairs['corr'][pairs.euc >= pairs.euc.quantile(0.75)].mean():.3f}")
d2 = TWO["842-221118"]
C2 = np.corrcoef(d2["Green"]["dff"].T)
print(f"Variant, session 2022-11-18: median r {np.median(C2[iu]):.3f}; r with distance {stats.pearsonr(C2[iu], pairs.euc)[0]:.3f}"
      f" (p = {stats.pearsonr(C2[iu], pairs.euc)[1]:.2g})")

**Result: reproduced.** Pairwise correlations of the 61 fibers are low (median r = 0.163, IQR -0.025
to 0.361) and variable (range -0.678 to 0.904), and fall with distance between tips (r = -0.190 across
1830 pairs, p = 2.8e-16; mean r 0.293 in the closest quarter of pairs against 0.133 in the farthest). The
other session gives the same (r = -0.238). The p-value treats pairs as independent, as the code does.

## figs2ef-no-bundle-crosstalk

> After accounting for distance in the brain, signal correlations do not depend on the distance between
> fibers on the bundle surface, so nearby fibers in the bundle do not cross-talk.

In [ ]:
X = sm.add_constant(pairs[["euc", "AP", "ML", "DV", "surface"]])
fit = sm.OLS(pairs["corr"], X).fit()
print(fit.summary2().tables[1].round(5).to_string())
print(f"\nbundle-surface distance (pixels): coefficient {fit.params['surface']:.2e} per pixel, 95% CI"
      f" {fit.conf_int().loc['surface', 0]:.2e} to {fit.conf_int().loc['surface', 1]:.2e}, p = {fit.pvalues['surface']:.3g}")
print(f"correlation of tip distance with surface distance: r = {stats.pearsonr(pairs.euc, pairs.surface)[0]:.3f}")
fit2 = sm.OLS(C2[iu], X).fit()
print(f"Variant, session 2022-11-18: surface coefficient {fit2.params['surface']:.2e}, p = {fit2.pvalues['surface']:.3g}")
near = pairs.surface <= pairs.surface.quantile(0.1)
print(f"Pairs adjacent on the bundle (closest 10%, <= {pairs.surface.quantile(0.1):.0f} px): mean r {pairs['corr'][near].mean():.3f},"
      f" others {pairs['corr'][~near].mean():.3f}")

**Result: reproduced.** In the paper's linear model, the distance between fibers on the bundle image
has no effect once tip distances are included (coefficient -3.4e-05 per pixel, 95% CI -2.7e-04 to
2.0e-04, p = 0.77), while tip distance does (Euclidean coefficient -0.278, p < 1e-5). The other session
gives p = 0.41. Fibers adjacent on the bundle correlate slightly more (0.222 against 0.156), which the
model attributes to their tips also being closer (bundle and tip distances correlate at r = 0.437).

## fig1d-chronic-reward-stability

> Dopamine release to unpredicted water reward is reliable and of comparable amplitude over at least 12
> weeks after implantation (median over 44 fibers, normalized to week 5).

In [ ]:
first = min(pd.Timestamp(s["start"]) for s in D15)


def weekly(sessions):
    rows = []
    for s in sessions:
        if len(s["act"]):
            wk = 5 + (pd.Timestamp(s["start"]) - first).days // 7
            rows.append((wk, s["act"]))
    weeks = sorted({w for w, _ in rows})
    means = {w: np.concatenate([a for ww, a in rows if ww == w]).mean(0) for w in weeks}
    counts = {w: sum(len(a) for ww, a in rows if ww == w) for w in weeks}
    init = means[weeks[0]][:31].max(0)
    tab = pd.DataFrame([{"week": w, "rewards": counts[w], "median": np.median(means[w][:31].max(0) / init),
                         "q25": np.percentile(means[w][:31].max(0) / init, 25), "q75": np.percentile(means[w][:31].max(0) / init, 75),
                         "median peak dff": np.median(means[w][:31].max(0))} for w in weeks])
    return tab


main = weekly([s for s in D15 if s["n_rew"] <= 25])
allr = weekly(D15)
print("Sessions with 25 rewards or fewer (main):")
print(main.round(3).to_string(index=False))
print("\nAll rewards of all sessions (variant):")
print(allr.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(main.week, main["median"], yerr=[main["median"] - main.q25, main.q75 - main["median"]], fmt="o-k", label="<= 25 rewards")
ax.errorbar(allr.week + 0.1, allr["median"], yerr=[allr["median"] - allr.q25, allr.q75 - allr["median"]], fmt="s--", c="0.5", label="all rewards")
ax.axhspan(0.7, 1.3, color="g", alpha=0.1)
ax.set_xlabel("week (first session = week 5)")
ax.set_ylabel("peak / week-5 peak, median over fibers")
ax.legend()
plt.show()
print("Plotted range of medians and quartiles: %.2f to %.2f" % (min(main.q25.min(), allr.q25.min()), max(main.q75.max(), allr.q75.max())))

**Result: partially reproduced.** In the sessions whose rewards are all unpredicted, the median peak
reward response of the 44 fibers stays between 0.90 and 1.20 of its week-5 value through week 8 (0.903,
0.932, 1.196), within the 0.7 to 1.3 band, but these sessions end at week 8; the 12 weeks of the claim
cannot be tested because the later sessions mix predicted and unpredicted rewards and the files do not say
which is which. With all rewards of all sessions the median falls to 0.77 in week 8, 0.61 in week 10 and
0.39 in week 13, which is expected if predicted rewards are included and does not test the claim.

## fig6-inclusion-counts

> In the mouse with ChR2 in dopamine axons and rDAm3.0 in the striatum, 24 recording locations lie within or
> on the border of the striatum; 18 of the 24 stimulated locations evoke a significant response at another
> location, and 22 of the 24 recording locations respond to at least one stimulation.

In [ ]:
rng_opto = np.random.default_rng(6060)
for o in OPTO:
    if o["stim"] is not None and len(o["off"]):
        o["resp"] = V.stim_response(o["dff"], o["on"], o["off"], o["fs"], rng_opto, laser_trace=o["dff"][:, o["stim"]])
o2 = [o for o in OPTO if o["subject"] == "Opto2" and o["n_artefact"] == 1]
inc = o2[0]["table"].included.values
print("Opto2 sessions with one stimulated fiber:", len(o2), "; left out:", [o["session"] for o in OPTO if o["subject"] == "Opto2" and o["n_artefact"] != 1])
print("Recording locations in or bordering the striatum (included):", int(inc.sum()))
stim_inc = [o for o in o2 if inc[o["stim"]]]
print("Sessions whose stimulated fiber is included:", len(stim_inc), "; stimulated fibers (1-based):", sorted(o["stim"] + 1 for o in stim_inc))
print("Stimulated fibers not included:", sorted(o["stim"] + 1 for o in o2 if not inc[o["stim"]]))


def counts(sessions, inc_mask):
    good_stim, responders = [], set()
    for o in sessions:
        sig = o["resp"]["sig"] & inc_mask
        others = sig.copy()
        others[o["stim"]] = False
        if others.sum() >= 1:
            good_stim.append(o)
        responders |= set(np.flatnonzero(sig))
    return good_stim, responders


good, responders = counts(stim_inc, inc)
print(f"\nStimulated locations with a significant response at another included location: {len(good)} of {len(stim_inc)}")
print(f"Included recording locations with a significant response to at least one stimulation: {len(responders)} of {int(inc.sum())}")
print("Per session: stimulated fiber, epochs, significant other fibers (increase, any):")
print(pd.DataFrame([{"session": o["session"], "stim": o["stim"] + 1, "epochs": len(o["off"]),
                     "sig up others": int((o["resp"]["sig_up"] & inc).sum() - o["resp"]["sig_up"][o["stim"]]),
                     "sig any others": int((o["resp"]["sig"] & inc).sum() - o["resp"]["sig"][o["stim"]]),
                     "stim fiber sig": bool(o["resp"]["sig"][o["stim"]])} for o in stim_inc]).to_string(index=False))
# Variant: the code's further exclusions (fibers 8, 9 of 35 um and broken fiber 21; 1-based)
inc_code = inc.copy()
inc_code[[7, 8, 20]] = False
good_c, resp_c = counts([o for o in stim_inc if inc_code[o["stim"]]], inc_code)
print(f"\nVariant, code's exclusions: locations {int(inc_code.sum())}; stimulated with a response elsewhere {len(good_c)} of"
      f" {len([o for o in stim_inc if inc_code[o['stim']]])}; responding recording locations {len(resp_c)}")
# Variant: fibers labelled striatal but not included (candidates for "on the border")
tabO = o2[0]["table"]
striatal = tabO.location.isin(["Caudoputamen", "Striatum", "Nucleus accumbens"]).values
inc_b = inc | striatal
print(f"Variant, included or labelled striatal: locations {int(inc_b.sum())} (added fibers {(np.flatnonzero(inc_b & ~inc) + 1).tolist()},"
      f" labels {tabO.location[inc_b & ~inc].tolist()})")
stim_b = [o for o in o2 if inc_b[o["stim"]]]
good_b, resp_b = counts(stim_b, inc_b)
print(f"   stimulated sessions {len(stim_b)}; stimulated with a response elsewhere {len(good_b)} of {len(stim_b)};"
      f" responding recording locations {len(resp_b)} of {int(inc_b.sum())}")

**Result: partially reproduced.** 22 Opto2 fibers are flagged included (expected 24 +- 2.4), and 20 of
them respond to at least one stimulation (expected 22 +- 2.2), both within tolerance; 14 of the 19
stimulated included fibers evoke a significant response at another included fiber, below the expected 18
+- 1.8. Counting the two fibers labelled Caudoputamen but not flagged (16 and 34) gives exactly the
paper's 24 locations and 22 responding ones, and 16 of 21 stimulated fibers with a response elsewhere; the
files hold 29 single-fiber sessions, of which only 21 stimulate a striatal fiber, so 18 of 24 cannot be
reached. The `included` flag appears to mark fibers within the striatum, not those on its border.

## fig6gh-release-decays-with-distance

> Stimulating dopamine axons through one fiber evokes release at neighbouring non-stimulated sites that drops
> off sharply with distance in all three dimensions.

In [ ]:
rng_fit = np.random.default_rng(6061)
tabO = o2[0]["table"]
pos = tabO[["ML", "AP", "DV"]].values
resp_fibers = np.array(sorted(responders))
pts = []
for o in good:
    for r in resp_fibers:
        if r != o["stim"]:
            dv = pos[r] - pos[o["stim"]]
            pts.append({"stim": o["stim"] + 1, "rec": r + 1, "resp": o["resp"]["resp"][r], "sig": bool(o["resp"]["sig"][r]),
                        "dML": dv[0], "dAP": dv[1], "dDV": dv[2], "dist": np.linalg.norm(dv)})
P6 = pd.DataFrame(pts)
print(f"points {len(P6)} ({P6.stim.nunique()} stimulation sites x {P6.rec.nunique()} recording sites); paper: 378 (18 x 21)")
a, b = V.exp_fit(P6.dist.values, P6.resp.values)
boot = []
sites = P6.stim.unique()
for _ in range(1000):
    pick = rng_fit.choice(sites, len(sites), replace=True)
    sub = pd.concat([P6[P6.stim == s] for s in pick])
    try:
        boot.append(V.exp_fit(sub.dist.values, sub.resp.values)[1])
    except RuntimeError:
        boot.append(np.nan)
boot = np.array(boot)
print(f"exponential fit: a = {a:.4f} DF/F, b = {b:.3f} per mm; 95% bootstrap interval of b {np.nanpercentile(boot, 2.5):.3f} to"
      f" {np.nanpercentile(boot, 97.5):.3f} (failed fits {int(np.isnan(boot).sum())} of 1000)")
for ax_ in ("dML", "dAP", "dDV"):
    rho, p = stats.spearmanr(P6[ax_].abs(), P6.resp)
    print(f"{ax_}: Spearman rho(|distance|, response) = {rho:.3f}, p = {p:.2g}")
print(f"response range {P6.resp.min():.4f} to {P6.resp.max():.4f}; significant points {int(P6.sig.sum())}; median distance of"
      f" significant {P6.dist[P6.sig].median():.2f} mm, of others {P6.dist[~P6.sig].median():.2f} mm")
bins = np.linspace(P6.dist.min(), P6.dist.max(), 12)
P6["bin"] = np.digitize(P6.dist, bins[1:-1])
print(P6.groupby("bin").agg(dist=("dist", "mean"), n=("resp", "size"), mean_resp=("resp", "mean")).round(4).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(P6.dist, P6.resp, s=8, c=np.where(P6.sig, "r", "0.6"))
xx = np.linspace(0, P6.dist.max(), 100)
ax.plot(xx, a * np.exp(b * xx), "k")
ax.set_xlabel("distance from stimulated fiber (mm)")
ax.set_ylabel("mean DF/F 0-0.5 s after offset")
plt.show()
print("All points shown; ranges above.")

**Result: reproduced.** Over 266 pairs of 14 stimulated and 20 recording fibers (the paper has 378,
18 by 21), the response falls with distance as a = 0.0147 DF/F, b = -1.096 per mm (95% bootstrap interval
-1.325 to -0.890), and with the absolute distance along each axis (Spearman rho -0.30 ML, -0.27 AP, -0.39
DV; all p < 1e-5). The binned mean falls from 0.0079 DF/F at 0.54 mm to 0.0013 to 0.0017 beyond 2 mm, and
significant responses lie closer (median 0.95 mm) than the others (1.57 mm).

## fig6g-dorsal-spread

> Evoked release drops off less with distance dorsal to the stimulated fiber than ventral to it,
> consistent with dorsally extending dopamine axon branches.

In [ ]:
def two_rates(df):
    out = {}
    for name, m in (("dorsal", df.dDV < 0), ("ventral", df.dDV > 0)):
        out[name] = V.exp_fit(df.dist[m].values, df.resp[m].values)[1]
    return out


rates = two_rates(P6)
print(f"dorsal sites {int((P6.dDV < 0).sum())}, ventral sites {int((P6.dDV > 0).sum())}")
print(f"decay rate b: dorsal {rates['dorsal']:.3f}, ventral {rates['ventral']:.3f} per mm")
diffs = []
for _ in range(1000):
    pick = rng_fit.choice(sites, len(sites), replace=True)
    sub = pd.concat([P6[P6.stim == s] for s in pick])
    try:
        rr = two_rates(sub)
        diffs.append(rr["dorsal"] - rr["ventral"])
    except RuntimeError:
        diffs.append(np.nan)
diffs = np.array(diffs)
print(f"difference dorsal - ventral {rates['dorsal'] - rates['ventral']:.3f}; 95% bootstrap interval {np.nanpercentile(diffs, 2.5):.3f}"
      f" to {np.nanpercentile(diffs, 97.5):.3f} (failed {int(np.isnan(diffs).sum())})")
for name, m in (("dorsal", P6.dDV < 0), ("ventral", P6.dDV > 0)):
    rho, p = stats.spearmanr(P6.dDV[m].abs(), P6.resp[m])
    print(f"{name}: Spearman rho(|dDV|, response) = {rho:.3f}, p = {p:.2g}; mean response {P6.resp[m].mean():.4f}")

**Result: partially reproduced.** Release decays more slowly dorsal to the stimulated fiber (b =
-0.971 per mm, 156 pairs) than ventral to it (b = -1.403, 109 pairs), but the 95% bootstrap interval of
the difference (-0.094 to 0.969) includes 0.

## fig6-tdtomato-control-no-response

> In control mice expressing tdTomato in place of rDAm3.0, single-fiber stimulation evokes no significant
> fluorescence increase at non-stimulated sites (0 of 48 sites, 26 stimulations in 2 mice), so the release
> is not a hemodynamic effect.

In [ ]:
rows = []
for o in OPTO:
    if o["subject"] not in ("Opto4", "Opto9", "Opto2") or "resp" not in o:
        continue
    if o["subject"] == "Opto2" and o["n_artefact"] != 1:
        continue
    inc_o = o["table"].included.values.copy()
    oth = inc_o.copy()
    oth[o["stim"]] = False
    up = o["resp"]["sig_up"] & oth
    rows.append({"subject": o["subject"], "session": o["session"], "stim": o["stim"] + 1, "stim included": bool(inc_o[o["stim"]]),
                 "artefact fibers": o["n_artefact"], "epochs": len(o["off"]), "non-stim included": int(oth.sum()),
                 "sig increase": int(up.sum()), "sig any": int((o["resp"]["sig"] & oth).sum()),
                 "max resp sig inc": round(float(o["resp"]["resp"][up].max()), 4) if up.any() else np.nan,
                 "median resp others": round(float(np.median(o["resp"]["resp"][oth])), 4)})
T6 = pd.DataFrame(rows)
print(T6[T6.subject != "Opto2"].to_string(index=False))
for s in ("Opto4", "Opto9", "Opto2"):
    t = T6[T6.subject == s]
    sites = set()
    for o in OPTO:
        if o["subject"] == s and "resp" in o and (s != "Opto2" or o["n_artefact"] == 1):
            oth = o["table"].included.values.copy()
            oth[o["stim"]] = False
            sites |= set(np.flatnonzero(o["resp"]["sig_up"] & oth))
    print(f"{s}: sessions {len(t)}; site-stimulation pairs tested {t['non-stim included'].sum()}; with a significant increase"
          f" {t['sig increase'].sum()}; distinct sites with one {len(sites)}; median response {t['median resp others'].median():.4f}")

**Result: reproduced.** In the tdTomato mice no included non-stimulated fiber shows a significant
increase after stimulation: 0 of 304 fiber and session pairs in Opto4 (13 sessions) and 0 of 203 in Opto9
(12 sessions), with median responses of -0.0002 and 0.0003 DF/F; one Opto9 pair has a significant
decrease. The same test finds increases in 142 of 619 pairs (20 distinct fibers) in Opto2. The files
hold 25 tdTomato sessions where the paper reports 26 stimulations. The stimulated fiber is taken as the
one with the largest laser artefact, which can be wrong when the artefact spreads; the true stimulated
fiber is then among those tested, which makes the test stricter.

## figs7a-photoswitch-confined

> In mice expressing rDAm3.0 without opsin, light through one fiber raises fluorescence at the stimulated
> site by photoswitching, but at no other site (0 of 22 non-stimulated sites for 3 stimulations).

In [ ]:
for o in [o for o in OPTO if o["subject"] == "UG21"]:
    inc_o = o["table"].included.values.copy()
    oth = inc_o.copy()
    oth[o["stim"]] = False
    r = o["resp"]
    print(f"UG21 {o['session']}: stimulated fiber {o['stim'] + 1} (included {bool(inc_o[o['stim']])}), epochs {len(o['off'])};"
          f" stimulated fiber: response {r['resp'][o['stim']]:.4f}, significant increase {bool(r['sig_up'][o['stim']])};"
          f" other included fibers {int(oth.sum())}: significant increase {int((r['sig_up'] & oth).sum())},"
          f" any {int((r['sig'] & oth).sum())}, median response {np.median(r['resp'][oth]):.4f}")

**Result: reproduced.** In each of UG21's 3 sessions the stimulated fiber (12, 18, 24) has a
significant increase after the laser (0.0118 to 0.0144 DF/F), and none of the 21 other included fibers
has a significant response of either sign (median responses -0.0035 to -0.0003).

## Claims not tested here

The data of these claims are not in the dandiset. The paper deposited them on Zenodo
([doi:10.5281/zenodo.10272874](https://doi.org/10.5281/zenodo.10272874)); this attempt did not test
them there.

- `figs1-no-tissue-damage`: NeuN and DAPI images of implanted and unimplanted striatum (SuppFig01.zip).
- `figs3-targeting-errors`: planned and localized fiber positions from the CT tables (SuppFig03.zip).
  The dandiset holds the localized tips but not the planned targets.
- `fig3-freely-moving-signals`: miniscope recordings in freely moving mice (Fig03.zip).
- `fig7d-ventral-not-dorsal-jaw` and `fig7ef-single-ventral-fiber`: the D1-cre mice OptoD1_6, 7, 8 and
  10, their DeepLabCut jaw tracking and stimulation files (Fig07.zip, SuppFig07.zip). None of these mice
  is in the dandiset.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1c-isosbestic-control | reproduced |
| figs2d-correlation-distance | reproduced |
| figs2ef-no-bundle-crosstalk | reproduced |
| fig1d-chronic-reward-stability | partially reproduced (weeks 5 to 8 only) |
| fig4-link-published-results (with inputs) | reproduced |
| fig4e-light-increases-widespread | reproduced |
| fig4e-light-anterior-posterior-gradient | partially reproduced |
| fig4f-tone-tail-increase-anterior-decrease | reproduced |
| fig4f-decreases-lateral-ventral | partially reproduced |
| figs4ac-patterns-consistent-across-mice | partially reproduced |
| figs4eh-patterns-not-movement | partially reproduced |
| figs5-isosbestic-no-pattern | partially reproduced |
| fig5ab-response-latencies | partially reproduced |
| fig5g-light-latency-gradient | reproduced |
| fig5h-tone-latency-gradient | reproduced |
| figs6cd-gradients-individual-mice | partially reproduced |
| fig6-inclusion-counts | partially reproduced |
| fig6gh-release-decays-with-distance | reproduced |
| fig6g-dorsal-spread | partially reproduced |
| fig6-tdtomato-control-no-response | reproduced |
| figs7a-photoswitch-confined | reproduced |
| figs1, figs3, fig3, fig7d, fig7ef | blocked: data only on Zenodo, not attempted |

**How the analysis differs from the paper's.** The GLM, its null and the latency rule follow the
Methods and the paper's code, re-implemented in Python: the low-pass filter is a Butterworth filter in
place of MATLAB's `lowpass`; the locomotion lags come from a cross-correlation that shifts negative lags,
where the code does not (the code's version is shown, r >= 0.989 with the main values); latencies are
counted from the onset frame as in the Methods, where the code counts one frame later. DL18's tones
include all 44 presentations, as the files do not give their frequency (its 14 12-kHz tones are used as a
variant). Stimulation times and stimulated fibers in the optogenetic sessions are found from the DF/F
alone, without the trigger line. The striatal tail boundary (-1.0 mm) and several criteria (thresholds
for "many", "low", "comparable") are choices made here, before the analysis.

**What was found about the dandiset.** It holds the data of Figures 1C-D, 4, 5, 6, S2D-F, S4, S5E-H,
S6 and S7A, and the traces, fiber order and stimulus times are exactly those of the paper's analysis.
Missing are: the stimulation times and the identity of the stimulated fiber in every optogenetic session;
the frequency of DL18's tones; the task variables that separate unpredicted rewards in DL15's later
sessions; one tdTomato session; the data of Figures 2, 3, 7, S1, S3 and S5A-D (on Zenodo). The `included`
column is described as "successfully implanted" but marks fibers within the striatum, leaving out
border fibers the optogenetic analysis counts; UG21's indicator is given as tdTomato where the paper's
counts and code identify it as the rDAm3.0 mouse without opsin; the archive's summary gives the approach
as cell population imaging; and the DL31 and DL32 series store a regular frame rate whose times drift by
up to half a second from the frame timestamps of the behaviour.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")